## 第 5 课：Tiled Copy 与全局内存访存

> 对应原文：笔记 (4)。从第一性原理解析 TiledCopy 的核心原理，并理解 NV GPU 的全局内存访存特性（向量化访存、合并访存）。

## 学习目标

- 理解 GMEM 访存的两种优化手段：**向量化访存**（单指令尽量长）与**合并访存**（warp 内尽量连续）；
- 理解 transaction / sector / cache line 的概念；
- 弄清上篇"LDG.E 冗余访存"问题的根因；
- 从原理上理解 TiledCopy：SrcLayout / DstLayout / RefLayout 与 partition_S / partition_D。

## 1. NV GPU 的全局内存访存特性

### 1.1 向量化访存

用更长字长的访存指令，减少指令调度、提升指令级并行。单指令最大访存量 **128 bits**：

In [ ]:
PTX                    SASS
32bit   ld.global.u32  LDG.E
64bit   ld.global.v2.u32  LDG.E.64
128bit  ld.global.v4.u32  LDG.E.128

Minimal GEMM 里 A 矩阵用两个 32bit LDG.E，是因为两块数据**不连续**。让单线程访存数据连续，编译器才会选用更长指令。

![使用 2 个 LDG.E 完成 A 矩阵分片的拷贝](assets/figs/fig_01_使用_2_个_LDG_E_完成_A_矩阵分片的拷贝.png)

### 1.2 合并访存

SIMT 架构下，warp 的 32 个线程同时执行一条访存指令，硬件把它合并为若干个 **transaction**。GMEM 的 transaction 最小单位是 32 bytes（**sector**）。访问不连续/不对齐、或横跨多个 sector 时，实际访存量 > 指令所需数据量（被访问 sector 的全部数据都会被读入缓存）。

SM60 及以上，无论是否经过 L1，transaction 固定 32 bytes；SM60 以下经过 L1 时是 128 bytes。

![各种访存情况示例](assets/figs/fig_02_各种访存情况示例.png)

### 1.3 Tiled MMA 的访存问题（上篇遗留）

以 A 矩阵第一个 warp 为例：T0-T31 的数据在 A(0,0)、A(0,1) 两行（每行恰好 1 个 sector），最优只需 16 个 sector。但每线程 4 块不连续数据 → 4 条 LDG.E，每条 LDG.E 都横跨 8 个 sector 却只用一半数据 → 实际读 32 个 sector，**多了一倍**。这就是 ncu 报"每条 LDG.E 有 50% 冗余访存"的原因。

![A 矩阵的 sector 分布](assets/figs/fig_04_A_矩阵的_sector_分布.png)  ![LDG.E 的访存不连续性](assets/figs/fig_05_LDG_E_的访存不连续性.png)

两种直觉方案都不可行：

- 让数据连续（调整 MMA Permutation 使同线程两块数据连续）→ 做不到；
- 固定 K 维度为 8 → 无法在 K 维扩展 MMA 规模。

彻底解决要引入 **SMEM**（下一篇）。

## 2. TiledCopy 的核心原理

所有拷贝归根到底是 `dst(i) = src(i)`：找到源地址和目标地址。CuTe 中数据以 Tensor（Data + Layout）存储，已知坐标就能算出地址，天然可以做拷贝：

In [ ]:
copy(dst, src);
// 等价于
for (int i = 0; i < size(src); ++i) {
    dst(i) = src(i);
}

![CuTe 中的 Tensor 拷贝基本原理](assets/figs/fig_06_CuTe_中的_Tensor_拷贝基本原理.png)

**关键假设**：CuTe 中参与 copy 的 src 和 dst 在相同坐标下对应同一份数据（恒等映射）。总可以通过变换 dst 的映射 `dst' = composition(dst, f)` 满足这一点。所以拷贝可以简化为 `dst(i) = src(i)`。

SIMT 下每个线程拿自己的分块，用 `(t, v)` 二元组索引：

In [ ]:
int t = threadIdx.x;
src_frg = src(t, _);
dst_frg = dst(t, _);
copy(dst_frg, src_frg);   // 每线程对自己的分块做循环拷贝

### 核心问题：如何构造 src 和 dst？

用两个 **TV Layout**：

- **Src TV Layout**：`(t, v) -> src 坐标`，描述每个线程从 src 读哪个坐标；
- **Dst TV Layout**：`(t, v) -> dst 坐标`，描述每个线程写 dst 的哪个坐标。

要让 `src(t,v)` 和 `dst(t,v)` 对应同一份数据，中间的桥梁**必须是数据本身**（ID 编号），而不是坐标/地址。这个桥梁由拷贝指令的本质特性决定，记录在 **CopyAtom** 中，即 **SrcLayout / DstLayout / RefLayout**。

- 常规拷贝指令（如 LDG/STG）：不交换线程间数据，SrcLayout 和 DstLayout 相同；
- 特殊指令（如 `ldmatrix`）：会做线程间数据交换，映射不再是恒等。

统一空间的方式：把 `(t,v)` 都映射到同一个 idx 空间（要么转成 src 空间、要么转成 dst 空间），选容易获取的那个作为 **Ref TV Layout**。

![src(t,v) 和 dst(t,v) 映射至同一个 idx](assets/figs/fig_08_src__t__v__和_dst__t__v__映射至同一个_idx.png)

![TiledCopy 的核心原理图](assets/figs/fig_09_TiledCopy_的核心原理图.png)

**结论**：TiledCopy 中 `partition_S` / `partition_D` 就是这个复合映射的工程实现。

## 3. Tiled Copy 实现

### 3.1 Copy_Traits 与 CopyAtom

拷贝指令的 Traits 记录三个映射。例：`SM75_U32x4_LDSM_N`（ldmatrix）：

In [ ]:
template <>
struct Copy_Traits<SM75_U32x4_LDSM_N> {
    using ThrID = Layout<_32>;
    using SrcLayout = Layout<Shape<_32, _128>, Stride<_128, _1>>;          // (src-thr,src-val) -> bit
    using DstLayout = Layout<Shape<_32, Shape<_32, _4>>,
                             Stride<_32, Stride<_1, _1024>>>;              // (dst-thr,dst-val) -> bit
    using RefLayout = DstLayout;                                           // 参考映射
};

CopyAtom 从 Traits 获取这些 Layout，并根据数据类型做处理（`recast_layout`），类似 MMA Atom 与 MMA Traits 的协作。

### 3.2 TiledCopy 与 ThrCopy

TiledCopy 在 CopyAtom 基础上扩展线程（T）和数据（V）两个维度。三个关键成员：

- `AtomLayoutSrc/Dst/Ref`：来自 CopyAtom 的 TV Layout；
- `Tiler_MN`：拷贝总规模（一个 Layout）；
- `TiledLayout_TV`：扩展后的 TV Layout。

`ThrCopy` 提供分块 API：

- `partition_S(src)`：把全局 src 经映射变为 src'，并取当前线程分块；
- `partition_D(dst)`：同理；
- `retile_S / retile_D`：在保持 size 不变的情况下重排 Tensor 的 Layout，适配拷贝的 shape。

In [ ]:
TiledCopyA g2r_tiled_copy_a;
ThrCopy g2r_thr_copy_a = g2r_tiled_copy_a.get_slice(tid);

Tensor tAgA = g2r_thr_copy_a.partition_S(gA);   // (CPY, CPY_M, CPY_K)
Tensor tAsA = g2r_thr_copy_a.partition_D(sA);   // (CPY, CPY_M, CPY_K)

注意：`(MMA, MMA_M, MMA_K)` 的 MMA 是**单个 Atom** 每线程的数据量；而 `(CPY, CPY_M, CPY_K)` 的 **CPY 是单个 Tile 每线程的总数据量**，CPY_M/CPY_K 是 Tile 扩展到 Block 的维度（未扩展前都是 1）。

### 3.3 make_tiled_copy API

In [ ]:
make_tiled_copy(CopyAtom, ThrLayout, ValLayout)

- ThrLayout：线程扩展方式；ValLayout：数据扩展方式；
- API 内部根据两者计算 Ref TV Layout 和拷贝规模 Tiler_MN。

当 Ref TV Layout 恰好是 TiledMMA 的 TV Layout（拷贝完立刻给 MMA 用）时，可以直接传 TiledMMA：

In [ ]:
using Copy_op = AutoVectorizingCopy;
using CopyA_atom = Copy_Atom<Copy_op, ComputeTypeA>;
using TiledCopyA = decltype(make_tiled_copy_A(CopyA_atom{}, TiledMMA{}));

![make_tiled_copy API](assets/figs/fig_05_make_tiled_copy_API.png)

### 3.4 算子代码示例

规格同笔记 (3)：问题规模 (32,32,16)，256 线程，GMEM→RMEM + MMA。

In [ ]:
// 创建四个矩阵的 TiledCopy（A/B 用 make_tiled_copy_A/B，C/O 用 C）
using TiledCopyA = decltype(make_tiled_copy_A(CopyA_atom{}, TiledMMA{}));
using TiledCopyB = decltype(make_tiled_copy_B(CopyB_atom{}, TiledMMA{}));
using TiledCopyC = decltype(make_tiled_copy_C(CopyC_atom{}, TiledMMA{}));
using TiledCopyO = decltype(make_tiled_copy_C(CopyO_atom{}, TiledMMA{}));

// kernel 内：retile 已有 TiledMMA 分块，交给 copy API
TiledCopyA g2r_tiled_copy_a;
ThrCopy g2r_thr_copy_a = g2r_tiled_copy_a.get_slice(tid);
Tensor tAgA = g2r_thr_copy_a.retile_S(tCgA);  // (CPY, CPY_M, CPY_K)
Tensor tArA = g2r_thr_copy_a.retile_D(tCrA);  // (CPY, CPY_M, CPY_K)
copy(g2r_tiled_copy_a, tAgA, tArA);

gemm(tiled_mma, tCrC, tCrA, tCrB, tCrC);      // 计算仍用 TiledMMA 分块

// 结果写回 GMEM（方向相反）
ThrCopy r2g_thr_copy_o = r2g_tiled_copy_o.get_slice(tid);
Tensor tCrC_r2g = r2g_thr_copy_o.retile_S(tCrC);
Tensor tCgC_r2g = r2g_thr_copy_o.retile_D(tCgC);
copy(r2g_tiled_copy_o, tCrC_r2g, tCgC_r2g);

### 3.5 metadata 解析与 latex 可视化

In [ ]:
cute::print(typename Spec::TiledCopyA{});
cute::print_latex(typename Spec::TiledCopyA{});

In [ ]:
TiledCopy
  Tiler_MN:       (_32,_16)
  TiledLayout_TV: ((_4,_8,_2,_4),((_2,_2,_2),(_1,_1))):((_64,_1,_16,_0),((_32,_8,_256),(_0,_0)))
Copy_Atom
  ThrID:        _1:_0
  ValLayoutSrc: (_1,_1):(_0,_0)
  ...

本例中 Src MN Layout 与 Dst MN Layout 完全相同（也等于 TiledMMA 的 A TV Layout）——说明拷贝没有线程间数据交换，每个线程只负责自己数据的拷贝。

![TiledCopy latex 可视化](assets/figs/fig_11_TiledCopy_latex_可视化.png)

## 同时回答

1. 向量化访存和合并访存分别解决什么问题？transaction / sector 各是什么？为什么 Tiled MMA 的每条 LDG.E 有 50% 冗余访存？
2. SrcLayout、DstLayout、RefLayout 分别是什么？为什么 `ldmatrix` 这类指令的 Src/Dst 映射不是恒等映射？
3. `(MMA, MMA_M, MMA_K)` 和 `(CPY, CPY_M, CPY_K)` 中 MMA 与 CPY 的含义有什么不同？`retile_S/D` 的作用是什么？

把代码和三个答案发给我，我继续审查。